# 04 — Perspective API Enrichment

Reads from the `posts_for_scoring` table (produced by notebook 05) and scores
every post with Google's Perspective API across 6 toxicity dimensions:

| Attribute | What it measures |
|-----------|------------------|
| `toxicity` | General rude/disrespectful tone |
| `severe_toxicity` | Hateful or threatening content |
| `identity_attack` | Negative stereotyping of identity groups |
| `insult` | Insulting or demeaning language |
| `profanity` | Swearing or obscene language |
| `threat` | Expressed intent to harm |

Results are stored in `perspective_scores` in `../data/moderation.db`.

> **Rate limit:** free tier = 1 QPS. 7,731 posts ≈ 2+ hours for full run.  
> Run in batches using `BATCH_SIZE` — the notebook resumes from where it left off.

In [2]:
import os
import sys
import sqlite3
import time
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

sys.path.insert(0, "..")
load_dotenv("../.env")

DB_PATH    = "../data/moderation.db"
API_URL    = "https://commentanalyzer.googleapis.com/v1alpha1/comments:analyze"
API_KEY    = os.getenv("PERSPECTIVE_API_KEY")
BATCH_SIZE = 100    # posts to score per run (change to None to score all)
DELAY      = 1.1   # seconds between requests (free tier: 1 QPS)

ATTRIBUTES = ["TOXICITY", "SEVERE_TOXICITY", "IDENTITY_ATTACK",
              "INSULT", "PROFANITY", "THREAT"]

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

if API_KEY:
    print(f"✓ API key loaded ({API_KEY[:8]}...)")
else:
    print("✗ PERSPECTIVE_API_KEY not found in .env")

✓ API key loaded (AIzaSyBe...)


## 1. Create perspective_scores table

In [3]:
conn = get_conn()
conn.executescript("""
    CREATE TABLE IF NOT EXISTS perspective_scores (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        platform          TEXT NOT NULL,
        post_id           TEXT NOT NULL,
        text_sample       TEXT,
        toxicity          REAL,
        severe_toxicity   REAL,
        identity_attack   REAL,
        insult            REAL,
        profanity         REAL,
        threat            REAL,
        scored_at         TEXT,
        UNIQUE(platform, post_id)
    );
    CREATE INDEX IF NOT EXISTS idx_persp_platform ON perspective_scores(platform);
    CREATE INDEX IF NOT EXISTS idx_persp_toxicity ON perspective_scores(toxicity);
""")
conn.commit()
conn.close()
print("✓ perspective_scores table ready")

✓ perspective_scores table ready


## 2. Check how many posts remain to score

In [4]:
conn = get_conn()

total   = conn.execute("SELECT COUNT(*) FROM posts_for_scoring").fetchone()[0]
already = conn.execute("SELECT COUNT(*) FROM perspective_scores").fetchone()[0]
remaining = total - already

print(f"Total posts in scoring table : {total:,}")
print(f"Already scored               : {already:,}")
print(f"Remaining                    : {remaining:,}")
print()

for row in conn.execute("""
    SELECT p.platform,
           COUNT(*) AS total,
           COUNT(ps.post_id) AS scored
    FROM posts_for_scoring p
    LEFT JOIN perspective_scores ps
           ON ps.post_id = p.post_id AND ps.platform = p.platform
    GROUP BY p.platform
"""):
    print(f"  {row['platform']:<10}: {row['scored']:>5} / {row['total']:>5} scored")

conn.close()
print(f"\nThis batch will score up to {BATCH_SIZE} posts (~{BATCH_SIZE*DELAY/60:.1f} min)")

Total posts in scoring table : 7,731
Already scored               : 0
Remaining                    : 7,731

  bluesky   :     0 /  6161 scored
  lemmy     :     0 /  1570 scored

This batch will score up to 100 posts (~1.8 min)


## 3. API smoke test

In [5]:
def score_text(text):
    """Call Perspective API. Returns dict of scores or None on error."""
    if not text or len(text.strip()) < 5:
        return None
    payload = {
        "comment": {"text": text[:3000]},
        "languages": ["en"],
        "requestedAttributes": {attr: {} for attr in ATTRIBUTES},
        "doNotStore": True,
    }
    try:
        resp = requests.post(API_URL, params={"key": API_KEY}, json=payload, timeout=15)
        if resp.status_code == 429:
            print("  Rate limited — waiting 60s")
            time.sleep(60)
            resp = requests.post(API_URL, params={"key": API_KEY}, json=payload, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        return {
            attr.lower(): data["attributeScores"][attr]["summaryScore"]["value"]
            for attr in ATTRIBUTES
            if attr in data.get("attributeScores", {})
        }
    except requests.HTTPError:
        print(f"  HTTP {resp.status_code}: {resp.text[:150]}")
        return None
    except Exception as e:
        print(f"  Error: {e}")
        return None

# Test
test = score_text("You are a terrible person and I hope something bad happens to you.")
if test:
    print("✓ Perspective API reachable")
    for attr, val in test.items():
        print(f"  {attr:<20}: {val:.3f}")
else:
    print("✗ API call failed — check key")

✓ Perspective API reachable
  toxicity            : 0.820
  severe_toxicity     : 0.354
  identity_attack     : 0.155
  insult              : 0.578
  profanity           : 0.345
  threat              : 0.603


## 4. Run scoring

Resumes automatically — skips posts already in `perspective_scores`.  
Change `BATCH_SIZE` at the top to score more/fewer posts per run.

In [6]:
conn = get_conn()

# Fetch unscored posts
limit_clause = f"LIMIT {BATCH_SIZE}" if BATCH_SIZE else ""
rows = conn.execute(f"""
    SELECT p.platform, p.post_id, p.text_clean
    FROM posts_for_scoring p
    WHERE NOT EXISTS (
        SELECT 1 FROM perspective_scores ps
        WHERE ps.platform = p.platform AND ps.post_id = p.post_id
    )
    {limit_clause}
""").fetchall()
conn.close()

print(f"Scoring {len(rows)} posts...")
print(f"Estimated time: {len(rows)*DELAY/60:.1f} minutes\n")

scored = skipped = errors = 0

for i, row in enumerate(rows):
    platform  = row["platform"]
    post_id   = row["post_id"]
    text      = row["text_clean"]

    scores = score_text(text)

    if scores:
        conn = get_conn()
        conn.execute("""
            INSERT OR IGNORE INTO perspective_scores
                (platform, post_id, text_sample, toxicity, severe_toxicity,
                 identity_attack, insult, profanity, threat, scored_at)
            VALUES (?,?,?,?,?,?,?,?,?,?)
        """, (
            platform, post_id, text[:200],
            scores.get("toxicity"),
            scores.get("severe_toxicity"),
            scores.get("identity_attack"),
            scores.get("insult"),
            scores.get("profanity"),
            scores.get("threat"),
            now_iso()
        ))
        conn.commit()
        conn.close()
        scored += 1
    else:
        errors += 1

    # Progress update every 10 posts
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(rows)}] scored={scored} errors={errors}", end="\r")

    time.sleep(DELAY)

print(f"\n\n✓ Done — scored: {scored} | errors: {errors}")

Scoring 100 posts...
Estimated time: 1.8 minutes

  [100/100] scored=100 errors=0

✓ Done — scored: 100 | errors: 0


## 5. Score distribution by platform

In [7]:
conn = get_conn()
df = pd.read_sql_query("""
    SELECT
        platform,
        COUNT(*)                       AS n_scored,
        ROUND(AVG(toxicity),3)         AS avg_toxicity,
        ROUND(AVG(severe_toxicity),3)  AS avg_severe,
        ROUND(AVG(identity_attack),3)  AS avg_identity,
        ROUND(AVG(insult),3)           AS avg_insult,
        ROUND(AVG(profanity),3)        AS avg_profanity,
        ROUND(AVG(threat),3)           AS avg_threat
    FROM perspective_scores
    GROUP BY platform
""", conn)
conn.close()
display(df)

,platform,n_scored,avg_toxicity,avg_severe,avg_identity,avg_insult,avg_profanity,avg_threat
0,bluesky,100,0.13,0.015,0.032,0.062,0.085,0.021


## 6. Toxicity threshold breakdown

In [8]:
conn = get_conn()
df_thresh = pd.read_sql_query("""
    SELECT
        platform,
        COUNT(*) AS total,
        SUM(CASE WHEN toxicity >= 0.8 THEN 1 ELSE 0 END) AS high_tox,
        SUM(CASE WHEN toxicity >= 0.5 AND toxicity < 0.8 THEN 1 ELSE 0 END) AS med_tox,
        SUM(CASE WHEN toxicity < 0.5 THEN 1 ELSE 0 END) AS low_tox,
        ROUND(100.0*SUM(CASE WHEN toxicity >= 0.8 THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_high
    FROM perspective_scores
    GROUP BY platform
""", conn)
conn.close()
print(df_thresh.to_string(index=False))

platform  total  high_tox  med_tox  low_tox  pct_high
 bluesky    100         0        5       95       0.0


## 7. BlueSky — scores joined with human labels

In [9]:
conn = get_conn()
df_joined = pd.read_sql_query("""
    SELECT
        SUBSTR(p.text_clean, 1, 80)    AS text_preview,
        l.label_val,
        ROUND(ps.toxicity, 3)          AS toxicity,
        ROUND(ps.severe_toxicity, 3)   AS severe_toxicity,
        ROUND(ps.identity_attack, 3)   AS identity_attack,
        ROUND(ps.insult, 3)            AS insult,
        ROUND(ps.profanity, 3)         AS profanity,
        ROUND(ps.threat, 3)            AS threat
    FROM perspective_scores ps
    JOIN bsky_posts_clean p  ON ps.post_id = p.uri AND ps.platform = 'bluesky'
    JOIN bsky_labels l       ON l.uri = p.uri
    ORDER BY ps.toxicity DESC
    LIMIT 10
""", conn)
conn.close()

pd.set_option("display.max_colwidth", 85)
display(df_joined)

,text_preview,label_val,toxicity,severe_toxicity,identity_attack,insult,profanity,threat
0,"I FORGOT TO PUT THIS HERE WHAT ANYWAYS BANG BANG MEME •°☆ BLOOD, FW & DED BODS 💀",graphic-media,0.285,0.023,0.028,0.044,0.189,0.138
1,saw a meme couldnot resist editing it for rt NO GRAPHIC MEDIA BUT HAS MAJOR SPOI,graphic-media,0.255,0.013,0.018,0.051,0.086,0.018
2,found a cute little spider this evening. think it's a jumping spider? added a gr,graphic-media,0.177,0.007,0.017,0.034,0.074,0.026
3,drifter and operator imgs Per Usual. very different (graphic media label for blo,graphic-media,0.113,0.005,0.011,0.021,0.035,0.054
4,"7.5 spoilers, not actually graphic media. I love this game so much sometimes, wh",graphic-media,0.097,0.004,0.009,0.021,0.038,0.012
5,🚨 Spoiler Alert for SWTOR Master's Enigma 🚨 Think this might be one of the best,graphic-media,0.063,0.003,0.008,0.015,0.030,0.012


## 8. Lemmy — scores joined with moderation reasons

In [10]:
conn = get_conn()
df_lemmy = pd.read_sql_query("""
    SELECT
        p.instance,
        SUBSTR(p.text_clean, 1, 80)    AS text_preview,
        p.reason,
        ROUND(ps.toxicity, 3)          AS toxicity,
        ROUND(ps.severe_toxicity, 3)   AS severe_toxicity,
        ROUND(ps.identity_attack, 3)   AS identity_attack
    FROM perspective_scores ps
    JOIN lemmy_posts_clean p
           ON ps.post_id = p.post_id || '@' || p.instance
          AND ps.platform = 'lemmy'
    WHERE p.reason IS NOT NULL AND p.reason != ''
    ORDER BY ps.toxicity DESC
    LIMIT 10
""", conn)
conn.close()

pd.set_option("display.max_colwidth", 85)
display(df_lemmy)

,instance,text_preview,reason,toxicity,severe_toxicity,identity_attack


## 9. Overall progress

In [11]:
conn = get_conn()
total     = conn.execute("SELECT COUNT(*) FROM posts_for_scoring").fetchone()[0]
scored    = conn.execute("SELECT COUNT(*) FROM perspective_scores").fetchone()[0]
remaining = total - scored
conn.close()

print(f"Total posts          : {total:,}")
print(f"Scored so far        : {scored:,} ({scored/total*100:.1f}%)")
print(f"Remaining            : {remaining:,}")
if remaining > 0:
    print(f"Est. time remaining  : {remaining*DELAY/60:.0f} minutes at 1 QPS")
    print()
    print("Re-run cell 4 to continue scoring. It resumes automatically.")
else:
    print("\n✓ All posts scored!")

Total posts          : 7,731
Scored so far        : 100 (1.3%)
Remaining            : 7,631
Est. time remaining  : 140 minutes at 1 QPS

Re-run cell 4 to continue scoring. It resumes automatically.
